# Lab 3 — Export to ONNX and serve with NVIDIA Triton

**Outcome:** export Parakeet CTC as a portable FP32 ONNX correctness baseline, place it in a Triton model repository, start Triton, and validate a transcript through the HTTP inference API. Estimated time: 45 minutes after the model is cached.

The model is exported in FP32 so the same numerically stable repository can move between T4, L4, A10 and A100. This is a GPU-served portability baseline, not a claim of maximum production throughput.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'voice_asr_lab').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('HF_HOME', str(ROOT / '.cache' / 'huggingface'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import subprocess, time
import numpy as np
import torch
from voice_asr_lab.audio import load_dummy_librispeech
from voice_asr_lab.asr import load_model_and_processor, load_trainable_state, processor_inputs
from voice_asr_lab.triton import export_fp32_onnx, infer_triton, write_triton_config


## 1. Load the model and optional Lab 2 state

If `artifacts/lab2_trainable_state.pt` exists, its changed parameters are applied before export. Otherwise the pretrained checkpoint is exported.

In [ ]:
records = load_dummy_librispeech(limit=1)
sample = records[0]
model, processor, _ = load_model_and_processor()
checkpoint = ROOT / 'artifacts' / 'lab2_trainable_state.pt'
if checkpoint.exists():
    load_trainable_state(model, checkpoint)
    print(f'Applied {checkpoint.name}')
else:
    print('Lab 2 checkpoint not found; exporting the pretrained model.')
sample_inputs = processor_inputs(processor, sample['audio'], sample['sampling_rate'])
with torch.inference_mode():
    pytorch_token_ids = model.generate(**sample_inputs)
pytorch_prediction = processor.batch_decode(pytorch_token_ids, skip_special_tokens=True)[0]
print(f'PyTorch reference: {pytorch_prediction}')

## 2. Export an FP32 ONNX graph

The source checkpoint uses BF16. A blanket FP16 conversion can overflow and produce NaN logits, so this workshop exports FP32 end to end and checks every returned logit before decoding. The time dimension is dynamic. Large ONNX weights may be stored beside the graph as external data; keep every generated file in the version directory.

In [ ]:
model_dir = ROOT / 'triton' / 'model_repository' / 'parakeet_ctc'
onnx_path = export_fp32_onnx(model, sample_inputs, model_dir / '1' / 'model.onnx')
config_path = write_triton_config(model_dir / 'config.pbtxt', model.config.vocab_size)
print(f'ONNX:   {onnx_path}')
print(f'Config: {config_path}')
print('Version files:', [p.name for p in sorted(onnx_path.parent.iterdir())])

In [ ]:
del model
torch.cuda.empty_cache()
print('Released the notebook model before Triton claims GPU memory.')

## 3. Start Triton

The first run downloads NVIDIA Triton 26.06. Ports 8000/8001/8002 remain internal to the Brev VM; only Jupyter needs a Secure Link.

In [ ]:
subprocess.run(['bash', str(ROOT / 'scripts' / 'start_triton.sh')], check=True)

## 4. Call the HTTP inference endpoint

Preprocessing remains in the client and Triton serves the neural network. Convert BF16 features to FP32 in PyTorch before calling NumPy, then verify that every Triton logit is finite before `argmax`. A production ensemble can move preprocessing and decoding server-side, but keeping the boundary visible is useful for this lab.

In [ ]:
features = sample_inputs['input_features'].detach().to(dtype=torch.float32).cpu().numpy()
mask = sample_inputs['attention_mask'].detach().to(dtype=torch.int64).cpu().numpy()
logits = infer_triton(features, mask)
finite_report = {
    'all_finite': bool(np.isfinite(logits).all()),
    'nan_count': int(np.isnan(logits).sum()),
    'inf_count': int(np.isinf(logits).sum()),
}
print(f'Finite logits: {finite_report}')
if not finite_report['all_finite']:
    raise RuntimeError('Triton returned NaN or Inf logits; do not decode or benchmark this artifact.')
token_ids = logits.argmax(axis=-1)
prediction = processor.batch_decode(token_ids, skip_special_tokens=True)[0]
transcript_match = prediction == pytorch_prediction
print(f'Reference:  {sample["text"]}')
print(f'PyTorch:    {pytorch_prediction}')
print(f'Triton:     {prediction}')
print(f'Match:      {transcript_match}')
print(f'Logits:     {logits.shape} {logits.dtype}')
if not prediction:
    raise RuntimeError('Triton produced an empty transcript; inspect token IDs and the exported graph.')
if not transcript_match:
    print('WARNING: Triton and PyTorch differ; inspect the transcript before treating this export as validated.')

In [ ]:
latencies = []
for _ in range(5):
    started = time.perf_counter()
    infer_triton(features, mask)
    latencies.append(time.perf_counter() - started)
print({
    'median_http_inference_seconds': float(np.median(latencies)),
    'p95_http_inference_seconds': float(np.percentile(latencies, 95)),
})

## Production handoff

The versioned FP32 Triton repository is the validated portability and correctness baseline. It remains GPU accelerated through ONNX Runtime and Triton, but uses roughly twice the weight and activation memory of a 16-bit artifact. For production, benchmark TensorRT or mixed precision against this baseline, retain FP32 for precision-sensitive operations when necessary, reject NaN or Inf outputs, compare held-out WER, and measure latency, throughput, and GPU memory on the target hardware. On EKS, put the selected repository in durable model storage, package/pin the Triton image, add readiness and liveness probes, expose a service, collect port 8002 metrics, define GPU requests, and load-test concurrency before choosing replica counts. NVIDIA Speech NIM is the supported packaged path when the required model/profile and NGC access fit the deployment.

**Checkpoint:** Triton reports ready, the finite-logit report contains zero NaN and Inf values, and the HTTP transcript matches the in-process path. When finished, run `bash scripts/stop_triton.sh` from a terminal to stop the lab container.